# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and performing simple analysis on the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

We will reference all record sets, fields, and columns via their Croissant `@id` for consistency.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We start by loading the Croissant dataset using its schema URL. We inspect the main metadata as provided by the Croissant model.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object; print name and description
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview

We will review available record sets and their associated fields (columns). All entities are referenced by their Croissant `@id`.

> ⚠️ mlcroissant exports Croissant metadata as objects. All record sets, fields, and columns are accessible from `dataset.metadata` by attribute.

Let's list all record sets with their id, name, and field ids:

In [ ]:
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
if not record_sets:
    print("No record sets found in metadata. Aborting.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']}")
        print(f"  name: {rs.get('name', 'No name')}\n  Fields (@id):")
        for field in rs.get('field', []):
            print(f"      - {field['@id']} ({field.get('name','')})")

## 3. Data Extraction

We will extract data from selected record sets using their `@id`. Update the `record_set_ids` variable to your record set(s) of interest as seen above.

In [ ]:
# ----
# Example code for extracting all records from a record set
# Adjust these @id values according to your exploration in section 2 above.
# ----
import warnings

# Example: find first recordSet @id
if not record_sets:
    raise ValueError("No record sets in dataset.metadata - cannot continue.")
example_record_set_id = record_sets[0]['@id']

# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Retrieve records for the record set
    try:
        # Each record is a dict with field @id as keys
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded record set: {record_set_id} ({len(df)} rows, {len(df.columns)} columns)")
    except Exception as e:
        warnings.warn(f"Failed loading {record_set_id}: {str(e)}")

# Show a sample of the first record set
first_df = dataframes[example_record_set_id]
print(f"Columns (@id) in first record set: \n", first_df.columns.tolist())
first_df.head()

## 4. Exploratory Data Analysis (EDA)

As an example, let's select a numeric field to filter and normalize, and potentially group by another field.
Please pick a field id from the data preview above for numeric/continuous data (e.g., coverage percent, peptide counts, mw, abundance, etc.), and optionally a category field id.

In [ ]:
# Replace these with the actual field ids for numeric and group fields.
# For this demo, we'll try to detect some plausible numeric field names from the columns.

df = dataframes[example_record_set_id]

import numpy as np

# Try to automatically find a numeric field id
numeric_candidate = None
for col in df.columns:
    # Very basic heuristic: try to convert values to numeric
    sample_values = df[col].dropna()[:20]
    try:
        as_num = pd.to_numeric(sample_values)
        if as_num.notnull().sum() >= 10:
            numeric_candidate = col
            break
    except Exception:
        continue

if numeric_candidate is None:
    raise Exception("Could not find a numeric field in the record set.")
print(f"Selected numeric field_id: {numeric_candidate}")

# Convert the entire column to numeric
df[numeric_candidate] = pd.to_numeric(df[numeric_candidate], errors='coerce')

# Filter: only keep rows where the value is greater than 10 (you may change threshold)
threshold = 10
filtered_df = df[df[numeric_candidate] > threshold]
print(f"Filtered records with {numeric_candidate} > {threshold}:")
print(filtered_df[[numeric_candidate]].head())

# Normalize this field
filtered_df[f"{numeric_candidate}_normalized"] = (
    filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()
) / filtered_df[numeric_candidate].std()
print(f"Normalized {numeric_candidate} for filtered records:")
print(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

# Find a group/categorical field to aggregate by
group_candidate = None
for col in df.columns:
    if col == numeric_candidate:
        continue
    nunique = df[col].nunique(dropna=True)
    if 2 <= nunique <= 10:
        group_candidate = col
        break

if group_candidate:
    grouped_df = filtered_df.groupby(group_candidate)[numeric_candidate].mean()
    print(f"Grouped data by {group_candidate} (mean {numeric_candidate}):")
    print(grouped_df.head())
else:
    print("No suitable group/categorical field found.")

## 5. Visualization

Let's plot the filtered, normalized data and examine its distribution.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(filtered_df) == 0:
    print("No records after filtering. Please adjust the threshold or choose a different field.")
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_candidate].dropna(), bins=30)
    plt.xlabel(numeric_candidate)
    plt.title(f"Distribution of {numeric_candidate} (filtered > {threshold})")
    plt.show()

    plt.figure(figsize=(8, 4))
    sns.boxplot(x=filtered_df[numeric_candidate])
    plt.title(f"Boxplot of {numeric_candidate} (filtered > {threshold})")
    plt.show()

    if group_candidate:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=filtered_df[group_candidate], y=filtered_df[numeric_candidate])
        plt.title(f"{numeric_candidate} distribution grouped by {group_candidate}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² Croissant dataset with `mlcroissant`.
- The main record sets and fields were discovered via their `@id`s.
- Sample numeric analysis included filtering, normalization, and (if possible) grouping by categorical variables.
- Plots provide an overview of value distributions. For further domain analysis, refer to the dataset's [FAIR² documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

You can extend this notebook for deeper protein analysis, differential expression, annotation cross-joins, etc., always referencing entities by their `@id` for reproducibility.